# Gradient Fluctations

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import matplotlib.pyplot as plt
import requests
from PIL import Image
import numpy as np
import time

# For interactive elements
from ipywidgets import interact, FloatSlider, IntSlider, fixed
from IPython.display import display

# Configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME = 'resnet50'
IMAGE_URL = 'https://raw.githubusercontent.com/utkuozbulak/pytorch-cnn-visualizations/master/input_images/cat_dog.png'
TARGET_CLASS = 242  # Boxer dog (ImageNet class index) - Adjust if your image shows a different dominant class
STEPS = 50 # Number of steps for the perturbation

# 1. Load pre-trained model
def load_model(name):
    model_class = models.__dict__[name]
    try:
        weights = model_class.Weights.DEFAULT
        model = model_class(weights=weights)
    except AttributeError:
        model = model_class(pretrained=True)
    return model

model = load_model(MODEL_NAME)
model = model.to(DEVICE).eval()

# 2. Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Denormalization helper (for visualization)
def denorm(img_tensor):
    img = img_tensor.squeeze(0).permute(1,2,0).cpu().detach().numpy()
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img * std + mean
    return np.clip(img, 0, 1)

# 3. Download and preprocess image
print("Downloading image...")
response = requests.get(IMAGE_URL, stream=True)
img = Image.open(response.raw)
input_tensor = preprocess(img).unsqueeze(0).to(DEVICE)

# Function to calculate gradient at a given input
def get_gradient(model, input_tensor, target_class):
    input_tensor.requires_grad_(True)
    model.zero_grad()
    output = model(input_tensor)
    target_score = output[0, target_class]
    target_score.backward()
    gradient = input_tensor.grad.data
    return gradient

# Interactive function for plotting gradient fluctuations, now with save_path option
def interactive_gradient_fluctuations_savable(noise_magnitude=0.05, num_sampled_pixels=3, save_path=None):
    original_image_tensor = input_tensor.clone().detach() # Detach to avoid interfering with later computations

    # Generate a random noise tensor
    epsilon = torch.randn_like(original_image_tensor) * noise_magnitude
    epsilon = epsilon.to(DEVICE)

    t_values = np.linspace(0, 1, STEPS)
    
    # Store gradients for selected pixel locations
    gradients_at_sampled_locations = []

    # Randomly select pixel locations for sampling
    H, W = original_image_tensor.shape[2], original_image_tensor.shape[3]
    
    selected_pixels = []
    # Seed numpy random for reproducibility of sampled pixels for a given run
    np.random.seed(42) # You can change or remove this seed if you want different pixels each time
    while len(selected_pixels) < num_sampled_pixels:
        r = np.random.randint(0, H)
        c = np.random.randint(0, W)
        if (r, c) not in selected_pixels:
            selected_pixels.append((r, c))
    
    # Only print this when not saving a static plot, to avoid spam in interactive sessions
    if save_path is None:
        print(f"Sampling gradients at pixel locations: {selected_pixels}")

    for t in t_values:
        # Create the perturbed input: x + t * epsilon
        perturbed_input = (original_image_tensor + t * epsilon).detach().requires_grad_(True)

        # Calculate the gradient of the target class score with respect to the perturbed input
        grad_at_t = get_gradient(model, perturbed_input, TARGET_CLASS)
        
        current_step_gradients = []
        for r, c in selected_pixels:
            pixel_gradients = grad_at_t[0, :, r, c].cpu().numpy()
            current_step_gradients.extend(pixel_gradients)
        gradients_at_sampled_locations.append(current_step_gradients)
    
    gradients_at_sampled_locations = np.array(gradients_at_sampled_locations) # Shape: (STEPS, num_sampled_pixels * 3)

    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    # Original Image
    axs[0].imshow(denorm(original_image_tensor))
    axs[0].set_title('Original Image')
    axs[0].axis('off')

    # Gradient Fluctuation Plot
    axs[1].set_xlabel('$t$')
    axs[1].set_ylabel(r'$\partial S_c / \partial x_i (x + t\epsilon)$')
    axs[1].set_title('Gradient Fluctuations along Perturbation Path')
    
    # Plotting each sampled pixel's R, G, B channel gradients
    colors = plt.cm.get_cmap('tab10', num_sampled_pixels * 3) # More distinct colors
    for i in range(gradients_at_sampled_locations.shape[1]): # Iterate over all (pixel, channel) combinations
        pixel_idx = i // 3
        channel_idx = i % 3
        axs[1].plot(t_values, gradients_at_sampled_locations[:, i], 
                     label=f'Pixel {pixel_idx+1} ({["R","G","B"][channel_idx]})',
                     color=colors(i))
    
    axs[1].axhline(y=0, color='black', linestyle='-') # Line at y=0
    axs[1].grid(True)
    # Set consistent y-limits to match original plot if desired, or let it auto-scale
    # Adjusted to allow for more visual fluctuation while keeping some consistency
    axs[1].set_ylim(-0.1, 0.1) 
    axs[1].ticklabel_format(style='sci', axis='y', scilimits=(0,0)) # Scientific notation for y-axis if needed

    # Noisy Image (at t=1)
    noisy_image_tensor = (original_image_tensor + 1 * epsilon).detach()
    axs[2].imshow(denorm(noisy_image_tensor))
    axs[2].set_title(f'Perturbed Image (t=1, Noise={noise_magnitude:.2f})')
    axs[2].axis('off')

    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path)
        plt.close(fig) # Close the figure to free memory if saving without showing
    else:
        plt.show() # Only show if not saving to a file directly

    return fig # Return the figure object (useful for `interact`)

if __name__ == '__main__':
    print(f"Using device: {DEVICE}")
    print(f"Model: {MODEL_NAME}")
    print(f"Target class: {TARGET_CLASS}")
    
    # --- Static PDF Generation ---
    # This section generates a single PDF with a fixed set of parameters.
    # It ensures the PDF is saved before any interactive plotting potentially clears the figure.
    pdf_filename = "results/gradient_fluctuations_interactive_example.pdf"
    print(f"\nGenerating a static PDF for a specific configuration: {pdf_filename}")
    # Call the savable function with a save_path
    interactive_gradient_fluctuations_savable(noise_magnitude=0.08, num_sampled_pixels=10, save_path=pdf_filename) 
    print(f"✅ Saved `{pdf_filename}`")

    # --- Interactive Plotting ---
    # This section sets up the interactive plot using ipywidgets.
    # This part should be run in a Jupyter Notebook, JupyterLab, or Google Colab environment.
    noise_slider = FloatSlider(
        value=0.05,
        min=0.001,
        max=0.1,
        step=0.005,
        description='Noise Magnitude:'
    )

    pixels_slider = IntSlider(
        value=3,
        min=1,
        max=10,
        step=1,
        description='Num Sampled Pixels:'
    )

    print("\nAdjust the sliders below to see real-time gradient fluctuations (run in Jupyter/Colab):")
    # Use interact to create the interactive plot.
    # `fixed(None)` ensures that `save_path` remains None during interactive updates,
    # so `plt.show()` is called for display.
    interact(interactive_gradient_fluctuations_savable, 
             noise_magnitude=noise_slider, 
             num_sampled_pixels=pixels_slider,
             save_path=fixed(None));

/Users/juliawenkmann/miniconda3/envs/ml/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/juliawenkmann/miniconda3/envs/ml/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Using device: cpu
Model: resnet50
Target class: 242

Generating a static PDF for a specific configuration: gradient_fluctuations_interactive_example.pdf


/var/folders/k7/xf0yhfbs02ggfm7n1l5g_cnm0000gn/T/ipykernel_61409/207799558.py:125: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colors = plt.cm.get_cmap('tab10', num_sampled_pixels * 3) # More distinct colors


✅ Saved `gradient_fluctuations_interactive_example.pdf`

Adjust the sliders below to see real-time gradient fluctuations (run in Jupyter/Colab):


interactive(children=(FloatSlider(value=0.05, description='Noise Magnitude:', max=0.1, min=0.001, step=0.005),…